# 03 - Modelado — DeepWeeds

**Proyecto:** Clasificación de especies de maleza en imágenes de campo mediante CNN y transfer learning

**Curso:** Fundamentos de Deep Learning — Universidad de Antioquia, 2026-2

**Autores:** Guillermo Mejía Uribe, Daniel Duque Rivera

### Qué cubre esta mitad

Este cuaderno entrena. Esta primera mitad deja lista la maquinaria y la comprueba, sin lanzar los doce entrenamientos de la primera ola.

1. Carga el contrato del cuaderno 02 y se niega a seguir si la firma del train no coincide.
2. Copia el pipeline de datos, sin cambiar el aumento ni el split.
3. Construye la CNN desde cero y las redes con pesos de ImageNet.
4. Entrena una cabeza con F1 macro medido sobre toda la validación, y rechaza el test y la etapa 2 en modo end-to-end.
5. Deja escrita, sin ejecutarla todavía, la época de depuración de la CNN plana.

### Qué queda para la segunda mitad

Entrenar ResNet-50, Inception-v3, EfficientNet-B0 y la CNN en las tres cabezas. Elegir el umbral en validación. Leer los errores. Tocar el test una vez.

### Preguntas que el cuaderno completo tiene que responder

1. En la tarea plana, ¿qué accuracy y qué F1 macro dejan los cuatro modelos del mismo protocolo?
2. En el mismo backbone, ¿la jerarquía cambia el F1 macro y los falsos positivos sobre Negative?
3. ¿El backbone mueve el resultado en otro sentido que la jerarquía?
4. Los errores que quedan, ¿son de la etapa 1, de la etapa 2, del par Chinee apple / Snake weed, del instrumento o del brillo?

La accuracy que se puede poner junto al 95,7 % de Olsen et al. (2019) es la de una red plana. La del sistema jerárquico va en otra fila. El 99,26 % de Duong et al. (2024) no es la meta.

El test no entra en `fit`, ni en el early stopping, ni en el umbral, ni en la elección del modelo.


## 1. Configuración

Los topes salen del plan y son iguales para todos los backbones. La rejilla del umbral se declara ya, aunque el barrido es de la segunda mitad: así no se abre después de ver el test.


In [1]:
import hashlib
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from sklearn.metrics import accuracy_score, f1_score

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

N_FOLDS = 5
FOLD_TRABAJO = 0
LADO_COMPARACION = 224
BATCH = 32
BATCH_DEFAULT = BATCH
N_CLASES = {"plana": 9, "etapa1": 2, "etapa2": 8}

# Misma política de aumento que el cuaderno 02. No se retoca aquí.
GIRO_MAXIMO_RAD = math.pi
ESCALA_MIN = 0.85
ESCALA_MAX = 1.15
BRILLO_ADITIVO = 25.0
BRILLO_ESCALA_MIN = 0.75
BRILLO_ESCALA_MAX = 1.25
DESPLAZAMIENTO_CANAL = 25.0

SUBCONJUNTOS = ["train", "val", "test"]
ETAPAS = ("plana", "etapa1", "etapa2")
MODOS = ("aislado", "e2e")
ESPECIES = {
    0: "Chinee apple",
    1: "Lantana",
    2: "Parkinsonia",
    3: "Parthenium",
    4: "Prickly acacia",
    5: "Rubber vine",
    6: "Siam weed",
    7: "Snake weed",
    8: "Negative",
}

# Topes del protocolo. Una llamada no puede pedir más épocas que estos.
EPOCAS_CABEZA = 8
EPOCAS_FINETUNE = 20
EPOCAS_SCRATCH = 25
LR_CABEZA = 1e-3
LR_FINETUNE = 1e-5
LR_SCRATCH = 1e-3
PACIENCIA_TL = 4
PACIENCIA_SCRATCH = 5
UMBRALES = (0.30, 0.40, 0.50, 0.60, 0.70)
PRIMERA_OLA = ("scratch", "ResNet50", "InceptionV3", "EfficientNetB0")

DIR_DATOS = Path("datos")
DIR_RESULTADOS = Path("resultados")
DIR_MODELOS = Path("modelos")
DIR_RESULTADOS.mkdir(exist_ok=True)
DIR_MODELOS.mkdir(exist_ok=True)

RUTA_MANIFIESTO = DIR_DATOS / "manifiesto_deepweeds.csv"
RUTA_NORM = DIR_DATOS / f"norm_scratch_fold{FOLD_TRABAJO}.json"
RUTA_REGISTRO = DIR_DATOS / "registro_backbones.json"


def exigir(condicion: bool, mensaje: str) -> None:
    """Detiene el cuaderno si un invariante no se cumple."""
    if not condicion:
        raise AssertionError(mensaje)


gpus = tf.config.list_physical_devices("GPU")
print(f"tensorflow {tf.__version__} | keras {keras.__version__}")
print("GPU:", [gpu.name for gpu in gpus] if gpus else "no hay, se usará CPU")
print(f"Rejilla de umbral, fija: {UMBRALES}")


tensorflow 2.21.0 | keras 3.15.1
GPU: no hay, se usará CPU
Rejilla de umbral, fija: (0.3, 0.4, 0.5, 0.6, 0.7)


## 2. Contrato del cuaderno 02

Se leen el manifiesto y los dos JSON. Si la firma del train no es la que quedó en la normalización, este cuaderno no entrena: el pipeline no sería el que ya se auditó.


In [2]:
import kagglehub


def firma_nombres(nombres) -> str:
    """SHA-256 de los filenames ordenados. La misma función que el cuaderno 02."""
    texto = "\n".join(sorted(map(str, nombres)))
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()


def descargar_dataset(id_dataset: str) -> Path:
    """Descarga un dataset de Kaggle con kagglehub."""
    try:
        return Path(kagglehub.dataset_download(id_dataset))
    except Exception as error:
        raise RuntimeError(
            "No se pudo localizar DeepWeeds. Hace falta el token en "
            "C:\\Users\\danie\\.kaggle\\access_token."
        ) from error


def localizar_carpetas(raiz: Path) -> tuple[Path, Path]:
    """Ubica las carpetas de etiquetas e imágenes a partir de labels.csv."""
    ruta_labels = next(raiz.rglob("labels.csv"), None)
    if ruta_labels is None:
        raise FileNotFoundError(f"No se encontró labels.csv en {raiz}")
    dir_labels = ruta_labels.parent
    dir_imagenes = dir_labels.parent / "images"
    if not dir_imagenes.is_dir():
        raise FileNotFoundError(f"No se encontró la carpeta images junto a {dir_labels}")
    return dir_labels, dir_imagenes


manifiesto = pd.read_csv(RUTA_MANIFIESTO)
NORMATIVAS = json.loads(RUTA_NORM.read_text(encoding="utf-8"))
REGISTRO = json.loads(RUTA_REGISTRO.read_text(encoding="utf-8"))["backbones"]

exigir(len(manifiesto) == 17509, "El manifiesto no tiene 17.509 filas.")
exigir(NORMATIVAS["fold"] == FOLD_TRABAJO, "La normalización es de otro fold.")
exigir(NORMATIVAS["lado"] == LADO_COMPARACION, "La normalización no es de lado 224.")
exigir(
    NORMATIVAS["usar_ponderacion"] == {"plana": True, "etapa1": False, "etapa2": False},
    "La política de pesos del JSON no es la del cuaderno 02.",
)
exigir(tuple(clave for clave in PRIMERA_OLA) == PRIMERA_OLA, "La primera ola cambió.")
for clave in PRIMERA_OLA:
    exigir(clave in REGISTRO, f"Falta {clave} en el registro.")
    exigir(int(REGISTRO[clave]["lado"]) == LADO_COMPARACION, f"{clave} no está en lado 224.")
    exigir(bool(REGISTRO[clave]["entrenar_en_la_primera_ola"]), f"{clave} no está en la primera ola.")

train = manifiesto.loc[manifiesto[f"split_{FOLD_TRABAJO}"] == "train", "Filename"]
positivas = manifiesto.loc[
    (manifiesto[f"split_{FOLD_TRABAJO}"] == "train") & (manifiesto["y_etapa1"].astype(int) == 1),
    "Filename",
]
exigir(
    firma_nombres(train) == NORMATIVAS["plana_etapa1"]["firma_nombres"],
    "La firma del train no coincide con la normalización. No se entrena.",
)
exigir(
    firma_nombres(positivas) == NORMATIVAS["etapa2"]["firma_nombres"],
    "La firma de las positivas de train no coincide. No se entrena.",
)
exigir(int(NORMATIVAS["plana_etapa1"]["n_imagenes"]) == 10501, "El train del fold 0 no tiene 10.501 imágenes.")
exigir(int(NORMATIVAS["etapa2"]["n_imagenes"]) == len(positivas), "La etapa 2 no coincide con las positivas del train.")

DIR_LABELS, DIR_IMAGENES = localizar_carpetas(descargar_dataset("imsparsh/deepweeds"))
print(f"Imágenes: {DIR_IMAGENES}")
print(f"Train {len(train)} | positivas de train {len(positivas)} | firmas en orden")
print("Pesos de la tarea plana (solo estos se aplican):")
display(pd.Series({int(k): v for k, v in NORMATIVAS["pesos"]["plana"].items()}).round(3).sort_index())


c:\Users\danie\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imágenes: C:\Users\danie\.cache\kagglehub\datasets\imsparsh\deepweeds\versions\2\images
Train 10501 | positivas de train 5038 | firmas en orden
Pesos de la tarea plana (solo estos se aplican):


0    1.731
1    1.829
2    1.888
3    1.903
4    1.832
5    1.929
6    1.812
7    1.916
8    0.214
dtype: float64

## 3. Pipeline heredado

Estas funciones son las del cuaderno 02. No se cambia el aumento, el orden de la normalización ni el criterio de las vistas. El dataset de entrenamiento entrega `(imagen, etiqueta, filename)`. `fit` no sabe qué hacer con el filename, así que la función de entrenamiento se queda solo con los dos primeros elementos. La validación del F1 sí usa la etiqueta y puede usar el nombre.


In [3]:
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_efficientnet
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inception
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenet
from tensorflow.keras.applications.resnet import preprocess_input as preprocess_resnet

PREPROCESS = {
    "ResNet50": preprocess_resnet,
    "InceptionV3": preprocess_inception,
    "EfficientNetB0": preprocess_efficientnet,
    "EfficientNetB4": preprocess_efficientnet,
    "MobileNetV2": preprocess_mobilenet,
}

def seleccionar_filas(manifiesto: pd.DataFrame, fold: int, split: str,
                      etapa: str, modo: str) -> pd.DataFrame:
    """Devuelve las filas y la etiqueta de una vista, sin leer píxeles.

    La etapa 2 aislada se queda solo con las positivas verdaderas: es el
    oracle, y es con lo que se entrena y se hace early stopping de esa cabeza.
    El modo end-to-end devuelve el split completo y la etiqueta plana, para
    que el sistema se evalúe sin filtrar el test con la respuesta correcta.

    Args:
        manifiesto: Tabla del cuaderno, una fila por imagen.
        fold: Fold oficial, de 0 a 4.
        split: 'train', 'val' o 'test'.
        etapa: 'plana', 'etapa1' o 'etapa2'.
        modo: 'aislado' o 'e2e'. El modo aislado solo existe para la etapa 2.

    Returns:
        Copia de las filas, con la columna 'y' que debe consumir el modelo.

    Raises:
        ValueError: Si la combinación de etapa y modo no está definida.
    """
    if split not in SUBCONJUNTOS:
        raise ValueError(f"Split desconocido: {split}")
    if etapa not in ETAPAS:
        raise ValueError(f"Etapa desconocida: {etapa}")
    if modo not in MODOS:
        raise ValueError(f"Modo desconocido: {modo}")
    if modo == "aislado" and etapa != "etapa2":
        raise ValueError(
            "El modo aislado solo aplica a la etapa 2. "
            "En la tarea plana y en la etapa 1 el split ya es el correcto."
        )
    if not 0 <= int(fold) < N_FOLDS:
        raise ValueError(f"Fold fuera de rango: {fold}")

    filas = manifiesto.loc[manifiesto[f"split_{fold}"] == split].copy()
    if etapa == "plana":
        filas["y"] = filas["Label"].astype(int)
    elif etapa == "etapa1":
        filas["y"] = filas["y_etapa1"].astype(int)
    elif modo == "e2e":
        filas["y"] = filas["Label"].astype(int)
    else:
        filas = filas.loc[filas["y_etapa1"].astype(int) == 1].copy()
        if filas.empty or filas["y_etapa2"].isna().any():
            raise ValueError("La etapa 2 aislada quedó sin etiquetas de especie.")
        filas["y"] = filas["y_etapa2"].astype(int)

    if filas["y"].isna().any():
        raise ValueError("La vista tiene etiquetas vacías.")
    return filas.reset_index(drop=True)

def leer_redimensionada(ruta: str, lado: int) -> tf.Tensor:
    """Lee un JPG y lo deja en float32 [0, 255] del lado indicado.

    No divide entre 255. Las funciones preprocess_input de Keras esperan
    esa escala y ellas mismas hacen la conversión.
    """
    imagen = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    imagen = tf.image.resize(imagen, [lado, lado], method="bilinear", antialias=True)
    return tf.cast(imagen, tf.float32)

def matriz_rotacion(angulo, alto, ancho) -> tf.Tensor:
    """Matriz proyectiva 1×8, la misma fórmula que Keras RandomRotation.

    Mapea el píxel de salida al píxel de entrada. Un ángulo 0 deja la imagen quieta.
    """
    angulo = tf.reshape(tf.cast(angulo, tf.float32), [1])
    alto = tf.cast(alto, tf.float32)
    ancho = tf.cast(ancho, tf.float32)
    coseno = tf.cos(angulo)
    seno = tf.sin(angulo)
    desplazamiento_x = ((ancho - 1) - (coseno * (ancho - 1) - seno * (alto - 1))) / 2.0
    desplazamiento_y = ((alto - 1) - (seno * (ancho - 1) + coseno * (alto - 1))) / 2.0
    ceros = tf.zeros([1, 2], dtype=tf.float32)
    return tf.concat([
        coseno[:, None], -seno[:, None], desplazamiento_x[:, None],
        seno[:, None], coseno[:, None], desplazamiento_y[:, None],
        ceros,
    ], axis=1)


def rotar_imagen(imagen: tf.Tensor, angulo) -> tf.Tensor:
    """Gira una imagen HWC. El borde se rellena por reflejo, no con negro."""
    imagen_4d = imagen[tf.newaxis, ...]
    alto = tf.shape(imagen_4d)[1]
    ancho = tf.shape(imagen_4d)[2]
    salida = tf.raw_ops.ImageProjectiveTransformV3(
        images=imagen_4d,
        transforms=matriz_rotacion(angulo, alto, ancho),
        output_shape=tf.cast(tf.stack([alto, ancho]), tf.int32),
        fill_value=tf.constant(0.0, tf.float32),
        interpolation="BILINEAR",
        fill_mode="REFLECT",
    )
    return salida[0]


def ajustar_al_tamano(imagen: tf.Tensor, alto, ancho) -> tf.Tensor:
    """Rellena por reflejo o recorta al centro hasta (alto, ancho)."""
    falta_alto = tf.maximum(alto - tf.shape(imagen)[0], 0)
    falta_ancho = tf.maximum(ancho - tf.shape(imagen)[1], 0)
    imagen = tf.pad(
        imagen,
        [[falta_alto // 2, falta_alto - falta_alto // 2],
         [falta_ancho // 2, falta_ancho - falta_ancho // 2],
         [0, 0]],
        mode="REFLECT",
    )
    inicio_alto = (tf.shape(imagen)[0] - alto) // 2
    inicio_ancho = (tf.shape(imagen)[1] - ancho) // 2
    return imagen[inicio_alto:inicio_alto + alto, inicio_ancho:inicio_ancho + ancho, :]


def escalar_y_recortar(imagen: tf.Tensor, factor) -> tf.Tensor:
    """Escala y devuelve al tamaño original. factor > 1 acerca; factor < 1 aleja."""
    alto = tf.shape(imagen)[0]
    ancho = tf.shape(imagen)[1]
    nuevo_alto = tf.maximum(tf.cast(tf.round(tf.cast(alto, tf.float32) * factor), tf.int32), 1)
    nuevo_ancho = tf.maximum(tf.cast(tf.round(tf.cast(ancho, tf.float32) * factor), tf.int32), 1)
    escalada = tf.image.resize(imagen, [nuevo_alto, nuevo_ancho], method="bilinear")
    return ajustar_al_tamano(escalada, alto, ancho)


def aumentar_imagen(imagen: tf.Tensor) -> tf.Tensor:
    """Aplica la política de aumento y recorta el resultado a [0, 255].

    Pensada solo para train. Dos llamadas sobre la misma imagen deben diferir.
    """
    angulo = tf.random.uniform([], -GIRO_MAXIMO_RAD, GIRO_MAXIMO_RAD)
    imagen = rotar_imagen(imagen, angulo)
    factor = tf.random.uniform([], ESCALA_MIN, ESCALA_MAX)
    imagen = escalar_y_recortar(imagen, factor)
    escala = tf.random.uniform([], BRILLO_ESCALA_MIN, BRILLO_ESCALA_MAX)
    aditivo = tf.random.uniform([], -BRILLO_ADITIVO, BRILLO_ADITIVO)
    canales = tf.random.uniform([1, 1, 3], -DESPLAZAMIENTO_CANAL, DESPLAZAMIENTO_CANAL)
    imagen = imagen * escala + aditivo + canales
    imagen = tf.image.random_flip_left_right(imagen)
    return tf.clip_by_value(imagen, 0.0, 255.0)


def brillo_medio(imagen: tf.Tensor) -> float:
    """Media de la luminancia, con los pesos usados en el cuaderno 01."""
    pesos = tf.constant([0.299, 0.587, 0.114], dtype=tf.float32)
    return float(tf.reduce_mean(tf.tensordot(imagen, pesos, axes=[[-1], [0]])))


rampa = tf.reshape(tf.linspace(0.0, 255.0, 32 * 32), [32, 32, 1])
rampa = tf.tile(rampa, [1, 1, 3])
giro_nulo = rotar_imagen(rampa, 0.0)
exigir(float(tf.reduce_max(tf.abs(giro_nulo - rampa))) < 1.0, "Un giro de 0° movió la imagen.")
giro_real = rotar_imagen(rampa, 0.7)
exigir(float(tf.reduce_max(tf.abs(giro_real - rampa))) > 10.0, "El giro no cambió la imagen.")
escala_nula = escalar_y_recortar(rampa, 1.0)
exigir(float(tf.reduce_max(tf.abs(escala_nula - rampa))) < 1.0, "Una escala de 1 cambió la imagen.")

def grupo_normalizacion(etapa: str) -> str:
    """Población de media y desviación que corresponde a la cabeza."""
    return "etapa2" if etapa == "etapa2" else "plana_etapa1"


def construir_dataset(tabla_manifiesto: pd.DataFrame, dir_imagenes: Path, fold: int,
                      split: str, etapa: str, modo: str, clave: str, entrenar: bool,
                      estadisticos: dict, batch_size: int = BATCH_DEFAULT) -> tf.data.Dataset:
    """Dataset de una vista, con aumento solo si se está entrenando el train.

    Args:
        tabla_manifiesto: Manifiesto del cuaderno.
        dir_imagenes: Carpeta de JPG.
        fold: Fold oficial.
        split: 'train', 'val' o 'test'.
        etapa: 'plana', 'etapa1' o 'etapa2'.
        modo: 'aislado' o 'e2e'.
        clave: Clave del registro de backbones.
        entrenar: Si es verdad, aplica aumento y baraja. Solo en train.
        estadisticos: Contenido de norm_scratch_fold{k}.json.
        batch_size: Tamaño de lote. No es una decisión de modelado cerrada aquí.

    Returns:
        Dataset de tuplas (imagen float32, etiqueta int32, filename).

    Raises:
        ValueError: Si se pide entrenar sobre val, test o sobre el end-to-end
            de la etapa 2.
        KeyError: Si el backbone no está en el registro.
    """
    if clave not in REGISTRO:
        raise KeyError(f"Backbone desconocido: {clave}")
    if entrenar and split != "train":
        raise ValueError("El aumento solo está definido para el split de train.")
    if entrenar and etapa == "etapa2" and modo == "e2e":
        raise ValueError(
            "La etapa 2 no se entrena con el split completo. "
            "Use modo='aislado' para entrenar y modo='e2e' solo para evaluar."
        )

    filas = seleccionar_filas(tabla_manifiesto, fold, split, etapa, modo)
    if filas.empty:
        raise ValueError(f"La vista {etapa}/{modo}/{split} quedó vacía.")

    lado = int(REGISTRO[clave]["lado"])
    rutas = [(dir_imagenes / nombre).as_posix() for nombre in filas["Filename"]]
    etiquetas = filas["y"].astype(int).tolist()
    nombres = filas["Filename"].tolist()

    if clave == "scratch":
        grupo = estadisticos[grupo_normalizacion(etapa)]
        media = tf.constant(grupo["media"], dtype=tf.float32)
        desviacion = tf.constant(grupo["std"], dtype=tf.float32)

        def normalizar(imagen):
            return (imagen - media) / desviacion
    else:
        funcion = PREPROCESS[clave]

        def normalizar(imagen):
            # preprocess_input espera 0–255. No anteponer x/255.
            return funcion(imagen)

    def cargar(ruta, etiqueta, nombre):
        imagen = leer_redimensionada(ruta, lado)
        if entrenar:
            imagen = aumentar_imagen(imagen)
        imagen = normalizar(imagen)
        imagen.set_shape([lado, lado, 3])
        return imagen, tf.cast(etiqueta, tf.int32), nombre

    dataset = tf.data.Dataset.from_tensor_slices((rutas, etiquetas, nombres))
    if entrenar:
        dataset = dataset.shuffle(len(filas), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(
        cargar,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=not entrenar,
    )
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def textos(tensor) -> list[str]:
    """Decodifica un tensor de strings de TensorFlow."""
    return [valor.decode("utf-8") for valor in tensor.numpy()]


## 4. Modelos

La CNN desde cero es la misma en las tres cabezas. Solo cambia la softmax: 9, 2 u 8. El transfer learning carga pesos de ImageNet, sin la cabeza original. La fase A deja la base quieta. La fase B la descongela.

EfficientNet está en modo `en_el_modelo`. No se le pone una capa de reescalado delante: la red ya normaliza por dentro. ResNet e Inception reciben los píxeles que `construir_dataset` ya pasó por `preprocess_input`.


In [4]:
def crear_cnn(n_clases: int) -> keras.Model:
    """Cuatro bloques Conv-BN-ReLU-MaxPool, pooling global y softmax.

    Es la línea base de la Entrega 1, no una arquitectura buscada después de ver el test.
    """
    entradas = keras.Input(shape=(LADO_COMPARACION, LADO_COMPARACION, 3))
    x = entradas
    for filtros in (32, 64, 128, 128):
        x = keras.layers.Conv2D(filtros, 3, padding="same", use_bias=False)(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.ReLU()(x)
        x = keras.layers.MaxPool2D()(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    salidas = keras.layers.Dense(n_clases, activation="softmax")(x)
    return keras.Model(entradas, salidas, name=f"scratch_{n_clases}")


CONSTRUCTORES = {
    "ResNet50": keras.applications.ResNet50,
    "InceptionV3": keras.applications.InceptionV3,
    "EfficientNetB0": keras.applications.EfficientNetB0,
}


def crear_transfer(clave: str, n_clases: int, congelar_base: bool) -> keras.Model:
    """Red de ImageNet sin la cabeza original, más una softmax nueva."""
    if REGISTRO[clave]["modo_rango"] == "en_el_modelo":
        # No añadir Rescaling aquí. Keras 3 ya lo trae dentro de EfficientNet.
        pass
    base = CONSTRUCTORES[clave](
        include_top=False,
        weights="imagenet",
        input_shape=(LADO_COMPARACION, LADO_COMPARACION, 3),
        pooling="avg",
    )
    base.trainable = not congelar_base
    x = keras.layers.Dropout(0.3)(base.output)
    salidas = keras.layers.Dense(n_clases, activation="softmax", name="cabeza")(x)
    return keras.Model(base.input, salidas, name=f"{clave}_{n_clases}")


def crear_modelo(clave: str, n_clases: int, congelar_base: bool) -> keras.Model:
    """Devuelve la CNN o una red con transfer learning."""
    if clave not in PRIMERA_OLA:
        raise KeyError(f"{clave} no está en la primera ola.")
    if clave == "scratch":
        return crear_cnn(n_clases)
    return crear_transfer(clave, n_clases, congelar_base)


def fijar_base(modelo: keras.Model, entrenable: bool) -> None:
    """Descongela o congela todo menos la dropout y la softmax finales."""
    for capa in modelo.layers[:-2]:
        capa.trainable = entrenable
    modelo.layers[-2].trainable = True
    modelo.layers[-1].trainable = True


def pesos_de_clase(etapa: str) -> dict[int, float] | None:
    """Pesos solo en la tarea plana. Las claves del JSON son texto."""
    if not NORMATIVAS["usar_ponderacion"][etapa]:
        return None
    return {int(clase): float(peso) for clase, peso in NORMATIVAS["pesos"][etapa].items()}


modelo_forma = crear_cnn(9)
salida_forma = modelo_forma(tf.zeros((2, LADO_COMPARACION, LADO_COMPARACION, 3)))
exigir(tuple(salida_forma.shape) == (2, 9), f"La CNN plana devolvió {tuple(salida_forma.shape)}.")
exigir(abs(float(tf.reduce_sum(salida_forma[0])) - 1.0) < 1e-4, "La softmax de la CNN no suma 1.")
del modelo_forma
print("CNN desde cero: entrada 224 y softmax de 9 en orden.")


CNN desde cero: entrada 224 y softmax de 9 en orden.


## 5. Entrenamiento de una cabeza

El F1 macro se calcula sobre la validación completa al final de la época. No se usa el promedio de los batches. La accuracy se anota y no elige el peso que se guarda.

La función solo abre train y val. No tiene un argumento para el test. La etapa 2 solo se acepta en modo `aislado`. Si se piden más épocas que el tope de esa fase, se detiene.


In [5]:
def tope_de(clave: str, fase: str) -> tuple[int, float, int]:
    """Épocas máximas, learning rate y paciencia de una fase."""
    if clave == "scratch":
        if fase != "entera":
            raise ValueError("La CNN desde cero se entrena en una sola fase, 'entera'.")
        return EPOCAS_SCRATCH, LR_SCRATCH, PACIENCIA_SCRATCH
    if fase == "A":
        return EPOCAS_CABEZA, LR_CABEZA, PACIENCIA_TL
    if fase == "B":
        return EPOCAS_FINETUNE, LR_FINETUNE, PACIENCIA_TL
    raise ValueError("El transfer learning usa la fase 'A' y después la fase 'B'.")


def dataset_de_fit(dataset: tf.data.Dataset) -> tf.data.Dataset:
    """Quita el filename. fit no debe interpretarlo como peso de muestra."""
    return dataset.map(lambda imagen, etiqueta, nombre: (imagen, etiqueta))


class F1Macro(keras.callbacks.Callback):
    """F1 macro de toda la validación. Guarda el mejor modelo y aplica la paciencia."""

    def __init__(self, dataset_val: tf.data.Dataset, paciencia: int, ruta_pesos: Path):
        super().__init__()
        self.dataset_val = dataset_val
        self.paciencia = paciencia
        self.ruta_pesos = Path(ruta_pesos)
        self.mejor = -1.0
        self.espera = 0
        self.historial: list[dict] = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        etiquetas = [lote[1].numpy() for lote in self.dataset_val]
        y_real = np.concatenate(etiquetas)
        probabilidades = self.model.predict(
            self.dataset_val.map(lambda imagen, etiqueta, nombre: imagen),
            verbose=0,
        )
        y_pred = np.argmax(probabilidades, axis=1)
        f1 = float(f1_score(y_real, y_pred, average="macro", zero_division=0))
        accuracy = float(accuracy_score(y_real, y_pred))
        logs["val_f1_macro"] = f1
        logs["val_accuracy_conjunto"] = accuracy
        self.historial.append({"epoca": int(epoch), "f1_macro": f1, "accuracy": accuracy})
        print(f"  época {epoch}: F1 macro val {f1:.4f} | accuracy val {accuracy:.4f}")
        if f1 > self.mejor:
            self.mejor = f1
            self.espera = 0
            self.model.save(self.ruta_pesos)
        else:
            self.espera += 1
            if self.espera >= self.paciencia:
                self.model.stop_training = True


def entrenar_cabeza(clave: str, etapa: str, modo: str, fase: str,
                    epocas: int | None = None, pasos: int | None = None,
                    modelo: keras.Model | None = None) -> tuple[keras.Model, list[dict]]:
    """Entrena una cabeza en train y elige la época con el F1 macro de val.

    Args:
        clave: Backbone de la primera ola.
        etapa: 'plana', 'etapa1' o 'etapa2'.
        modo: 'e2e' o, solo en la etapa 2, 'aislado'.
        fase: 'A', 'B' o 'entera'.
        epocas: Si se pasa, no puede superar el tope de la fase.
        pasos: Si se pasa, recorta el número de batches de train. La validación sigue completa.
        modelo: Modelo ya construido. La fase B reutiliza el de la fase A.

    Returns:
        El modelo de la mejor época y el historial de validación.

    Raises:
        ValueError: Si la etapa 2 no va en modo aislado, o si la fase no corresponde
            al tipo de red, o si se piden más épocas que el tope.
    """
    if etapa == "etapa2" and modo != "aislado":
        raise ValueError(
            "La etapa 2 no se entrena con el split completo. Use modo='aislado'."
        )
    if modo == "aislado" and etapa != "etapa2":
        raise ValueError("El modo aislado solo existe para la etapa 2.")
    tope, lr, paciencia = tope_de(clave, fase)
    if epocas is None:
        epocas = tope
    if epocas > tope:
        raise ValueError(f"La fase {fase} de {clave} tiene un tope de {tope} épocas.")

    n_clases = N_CLASES[etapa]
    if modelo is None:
        modelo = crear_modelo(clave, n_clases, congelar_base=(fase == "A"))
    if clave != "scratch":
        fijar_base(modelo, entrenable=(fase == "B"))
    modelo.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
    )

    ds_train = construir_dataset(
        manifiesto, DIR_IMAGENES, FOLD_TRABAJO, "train", etapa, modo,
        clave, True, NORMATIVAS, batch_size=BATCH,
    )
    ds_val = construir_dataset(
        manifiesto, DIR_IMAGENES, FOLD_TRABAJO, "val", etapa, modo,
        clave, False, NORMATIVAS, batch_size=BATCH,
    )
    ruta = DIR_MODELOS / f"{clave}_{etapa}_{fase}.keras"
    callback = F1Macro(ds_val, paciencia, ruta)
    modelo.fit(
        dataset_de_fit(ds_train),
        epochs=epocas,
        steps_per_epoch=pasos,
        class_weight=pesos_de_clase(etapa),
        callbacks=[callback],
        verbose=1,
    )
    if ruta.is_file():
        modelo = keras.models.load_model(ruta)
    bitacora = {
        "clave": clave,
        "etapa": etapa,
        "modo": modo,
        "fase": fase,
        "epocas_pedidas": epocas,
        "mejor_f1_macro": callback.mejor,
        "historial": callback.historial,
        "pesos": str(ruta),
    }
    ruta_json = DIR_RESULTADOS / f"{clave}_{etapa}_{fase}.json"
    ruta_json.write_text(json.dumps(bitacora, indent=2), encoding="utf-8")
    print(f"Bitácora: {ruta_json}")
    return modelo, callback.historial


try:
    entrenar_cabeza("scratch", "etapa2", "e2e", "entera", epocas=1)
except ValueError as error:
    print(f"Rechazada, como debe: {error}")
else:
    raise AssertionError("Se permitió entrenar la etapa 2 con el split completo.")

try:
    entrenar_cabeza("ResNet50", "plana", "e2e", "A", epocas=EPOCAS_CABEZA + 1)
except ValueError as error:
    print(f"Rechazada, como debe: {error}")
else:
    raise AssertionError("Se permitió pasar el tope de épocas.")

pesos_plana = pesos_de_clase("plana")
exigir(pesos_plana is not None and set(pesos_plana) == set(range(9)), "La tarea plana no tiene 9 pesos.")
exigir(pesos_de_clase("etapa1") is None and pesos_de_clase("etapa2") is None,
       "La jerarquía no debe llevar pesos de clase.")
print("Guardas de entrenamiento en orden. El test no es un argumento de entrenar_cabeza.")


Rechazada, como debe: La etapa 2 no se entrena con el split completo. Use modo='aislado'.
Rechazada, como debe: La fase A de ResNet50 tiene un tope de 8 épocas.
Guardas de entrenamiento en orden. El test no es un argumento de entrenar_cabeza.


## 6. Época de depuración

Esta celda recorre **una** época de la CNN plana, con el train completo y el F1 macro de toda la validación. Sirve para ver que las formas, los pesos y el callback cierran. No elige modelo y no toca el test.

En CPU puede tardar bastante. No la lances si solo quieres revisar el código: las celdas anteriores ya comprobaron las guardas y la forma de la red.

La segunda mitad, todavía no escrita en celdas ejecutables, es esta:

1. ResNet-50 plana, luego su etapa 1 y su etapa 2, con el umbral en la rejilla de validación.
2. Inception-v3 y EfficientNet-B0, las tres cabezas, con los mismos topes y el mismo Adam.
3. La CNN desde cero, las tres cabezas, hasta el tope de 25 épocas.
4. Elegir el modelo plano por F1 macro de validación y el sistema jerárquico por F1 macro end-to-end de validación.
5. Leer en validación el par Chinee apple / Snake weed, el instrumento y el brillo, sin cambiar el umbral.
6. Evaluar el test una vez y escribir `resultados/fold0_cierre.json` antes de la lectura contra Olsen y Duong.


In [6]:
# Primera época real de la CNN plana. Quita el comentario de la llamada cuando vayas a correrla.
# En CPU recorre las 10.501 imágenes de train y las 3.501 de validación.
#
# modelo_depuracion, historial_depuracion = entrenar_cabeza(
#     "scratch", "plana", "e2e", "entera", epocas=1,
# )
print(
    "Depuración lista y sin lanzar. "
    "Descomenta la llamada de esta celda para la única época de comprobación."
)


Depuración lista y sin lanzar. Descomenta la llamada de esta celda para la única época de comprobación.
